In [ ]:
import pickle
import torch
import openai

openai.api_key= ...

In [3]:
# KG laden
with open('final_kg.pkl', 'rb') as f:
    G = pickle.load(f)

node_list = list(G.nodes())
print("Anzahl Knoten im neuen KG:", len(node_list))
print("Beispiel-Knoten:", node_list[:5])

# Embeddings laden
embeddings = torch.load('final_kg_embeddings_tensor.pt')
print("Embeddings-Shape:", embeddings.shape)

Anzahl Knoten im neuen KG: 161005
Beispiel-Knoten: [16687, 33577, 36035, 20297, 16693]


/var/folders/bb/qyhd42qs7tj0cdb4wpzgsq1r0000gn/T/ipykernel_77630/953882501.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  embeddings = torch.load('final_kg_embeddings_

Embeddings-Shape: torch.Size([161005, 1536])


In [11]:
import os
import pickle
import torch
from tqdm import tqdm
import openai  # oder dein Embedding-Modell
import time

with open('final_kg.pkl', 'rb') as f:
    G = pickle.load(f)
node_list = list(G.nodes())
print("Knoten im KG:", len(node_list))

# === 3. Embedding-Funktion mit Batching und Fehlerbehandlung ===
def get_embeddings_batch(texts, model="text-embedding-ada-002", max_retries=5):
    for attempt in range(max_retries):
        try:
            response = openai.Embedding.create(input=texts, model=model)
            return [torch.tensor(item['embedding'], dtype=torch.float32) for item in response['data']]
        except openai.error.RateLimitError:
            print("Rate limit erreicht, warte 10 Sekunden...")
            time.sleep(10)
        except Exception as e:
            print(f"Fehler: {e}, versuche es erneut in 5 Sekunden...")
            time.sleep(5)
    raise RuntimeError("Zu viele Fehler bei OpenAI-API!")

# === 4. Embeddings erzeugen (mit Batching und Zwischenspeichern) ===
BATCH_SIZE = 50
embeddings = []
start_idx = 0

# Falls du fortsetzen willst:
if os.path.exists('final_kg_embeddings_tensor.pt'):
    print("Lade bereits gespeicherte Embeddings...")
    embeddings = torch.load('final_kg_embeddings_tensor.pt')
    start_idx = len(embeddings)
    embeddings = list(embeddings)
    print(f"Setze ab Index {start_idx} fort.")

try:
    for i in tqdm(range(start_idx, len(node_list), BATCH_SIZE)):
        batch_nodes = node_list[i:i+BATCH_SIZE]
        texts = [G.nodes[node].get('name', str(node)) for node in batch_nodes]
        batch_embeddings = get_embeddings_batch(texts)
        embeddings.extend(batch_embeddings)
        # Zwischenspeichern nach jedem Batch
        if (i // BATCH_SIZE) % 10 == 0:
            torch.save(torch.stack(embeddings), 'final_kg_embeddings_tensor.pt')
            print(f"Zwischenspeicher: {len(embeddings)} Embeddings gespeichert.")
    # Am Ende alles speichern
    torch.save(torch.stack(embeddings), 'final_kg_embeddings_tensor.pt')
    print("Alle Embeddings gespeichert!")
except Exception as e:
    print("Abbruch wegen Fehler:", e)
    torch.save(torch.stack(embeddings), 'final_kg_embeddings_tensor.pt')
    print("Bisherige Embeddings gespeichert.")

print("Fertig!")

Knoten im KG: 161005


  0%|          | 1/3221 [00:00<37:50,  1.42it/s]

Zwischenspeicher: 50 Embeddings gespeichert.


  0%|          | 11/3221 [00:06<34:12,  1.56it/s]

Zwischenspeicher: 550 Embeddings gespeichert.


...

100%|█████████▉| 3211/3221 [33:37<00:07,  1.39it/s]

Zwischenspeicher: 160550 Embeddings gespeichert.


100%|██████████| 3221/3221 [33:44<00:00,  1.59it/s]

Zwischenspeicher: 161005 Embeddings gespeichert.


Alle Embeddings gespeichert!
Fertig!


In [4]:
emb_path = '../../util/datasets/final_kg_embeddings_tensor.pt'
embeddings = torch.load(emb_path)
print(type(embeddings))
if isinstance(embeddings, dict):
    print("Keys:", list(embeddings.keys())[:5])
elif isinstance(embeddings, list):
    print("List length:", len(embeddings))
elif hasattr(embeddings, 'shape'):
    print("Shape:", embeddings.shape)
else:
    print("Unbekannter Typ:", embeddings)

/var/folders/bb/qyhd42qs7tj0cdb4wpzgsq1r0000gn/T/ipykernel_77630/2434782612.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  embeddings = torch.load(emb_path)


<class 'torch.Tensor'>
Shape: torch.Size([161005, 1536])


In [5]:
import pickle
import torch

# Pfade anpassen!
kg_path = '../../util/datasets/final_kg.pkl'
emb_path = '../../util/datasets/final_kg_embeddings_tensor.pt'
output_path = '../../util/datasets/final_kg_with_embeddings.pkl'

# Graph laden
with open(kg_path, 'rb') as f:
    kg = pickle.load(f)

# Embeddings laden
embeddings = torch.load(emb_path)

# Prüfen, ob die Anzahl der Knoten und Embeddings übereinstimmt
nodes = list(kg.nodes())
if len(nodes) != embeddings.shape[0]:
    print(f"Achtung: {len(nodes)} Knoten, aber {embeddings.shape[0]} Embeddings!")
else:
    print("Knoten und Embeddings passen zusammen.")

# Embeddings zuweisen
for idx, node in enumerate(nodes):
    kg.nodes[node]['embedding'] = embeddings[idx].numpy()  # als numpy-Array speichern

# Graph abspeichern
with open(output_path, 'wb') as f:
    pickle.dump(kg, f)

print(f"Fertig! Neuer Graph mit Embeddings gespeichert unter: {output_path}")

/var/folders/bb/qyhd42qs7tj0cdb4wpzgsq1r0000gn/T/ipykernel_77630/36924246.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  embeddings = torch.load(emb_path)


Knoten und Embeddings passen zusammen.
Fertig! Neuer Graph mit Embeddings gespeichert unter: ../../util/datasets/final_kg_with_embeddings.pkl
